# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa — Exploration with `mlcroissant`
This notebook provides a practical guide for loading, exploring, and processing the FAIR² mental health survey dataset from Kilifi County, Kenya, using the `mlcroissant` library. All data entities are referenced by their Croissant `@id`.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"{metadata.name}\n\n{metadata.description}")
print("\nDataset citation:")
print(metadata.cite_as)

## 2. Data Overview
Review available record sets, fields, and their IDs.

mlcroissant provides metadata browsing of record sets and fields. All entities are referenced by their `@id`.

In [ ]:
# List all record sets (by @id)
record_sets = metadata.record_sets
print(f"Record sets found: {len(record_sets)}\n")
for rs in record_sets:
    print(f"@id: {rs.id} | Name: {rs.name}")

# For each record set, display its available fields (by @id)
for rs in record_sets:
    print(f"\nRecord set: {rs.name} | @id: {rs.id}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    @id: {field.id} | Name: {field.name} | Type: {field.data_type}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. 

_**All data elements are referenced by their `@id` values as shown above.**_

In [ ]:
# Choose one or more record sets for extraction (replace with the chosen @id from the list above)
record_set_ids = [rs.id for rs in metadata.record_sets]

dataframes = {}

# Load all record sets into dataframes
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# Display the first DataFrame structure as an example
main_record_set = record_set_ids[0]  # Use the first record set as the main table for demo
print(f"Fields in record set '{main_record_set}':")
print(dataframes[main_record_set].columns.tolist())
dataframes[main_record_set].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records by criteria, normalizing numeric fields, and grouping. 

_All field and group columns referenced by their exact Croissant `@id`._

In [ ]:
# Example: Try analyzing the PHQ-9 total score (id may be 'phq9_total_score'), and group by gender if available.

# Replace these with actual field @id values as discovered above
numeric_field_id = None
group_field_id = None
for field in metadata.record_sets[0].fields:
    # Try to find a PHQ-9 or GAD-7 total score field
    if ("phq" in field.id.lower() or "gad" in field.id.lower() or "psq" in field.id.lower()) and ("total" in field.id.lower() or "score" in field.id.lower()):
        numeric_field_id = field.id
    if "gender" in field.id.lower() or "sex" in field.id.lower():
        group_field_id = field.id

if numeric_field_id is None:
    print("No numeric field like PHQ-9/GAD-7 found! Please check the output above for available field @id's.")
else:
    print(f"Using numeric field: {numeric_field_id}")
    df = dataframes[main_record_set]
    # Drop missing values to avoid errors
    filtered_df = df[df[numeric_field_id].notnull()].copy()
    threshold = filtered_df[numeric_field_id].mean() + filtered_df[numeric_field_id].std()
    filtered_df = filtered_df[filtered_df[numeric_field_id] > threshold]
    print(f"Filtered records where {numeric_field_id} > mean+1 std")
    print(filtered_df.head())

    # Normalize
    col_norm = f"{numeric_field_id}_normalized"
    filtered_df[col_norm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, col_norm]].head())

    # Group
    if group_field_id is not None and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"\nMean {numeric_field_id} by {group_field_id} (for records > threshold):")
        print(grouped_df)
    else:
        print("No gender/sex field found for grouping.")

## 5. Visualization
Visualize the distribution and relationships of key fields. Example shown for a total score, but you can adapt for other fields as desired.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None:
    df = dataframes[main_record_set]
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id} (raw values)")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If group field present, plot by group
    if group_field_id is not None and group_field_id in df.columns:
        plt.figure(figsize=(8,5))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()


## 6. Conclusion
In this notebook, we've demonstrated loading, overviewing, filtering, and visualizing the FAIR² Mental Health Survey Dataset using `mlcroissant`. 

- All schema entities were referenced by their Croissant `@id`.
- Data extraction is fully dynamic and robust to schema changes.
- You can continue with more advanced analysis by referencing any new columns or record sets using their corresponding `@id`.

_To cite this dataset: Mugotitsa, B, et al, 2026. A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya. Frontiers._
